# 第五章

## 层和块


In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
#自定义块
class MLP(nn.Module):
    # 用模型参数声明层。声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        #在类实例化时也可以指定其他函数参数，如模型参数params
        super().__init__()
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.out = nn.Linear(256, 10)  # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 使用ReLU的函数版本，其在nn.functional模块中定义。
        return self.out(F.relu(self.hidden(X)))

In [4]:
net = MLP()
X = torch.rand(2, 20)
net(X)

tensor([[ 0.2972, -0.1446, -0.1449,  0.0981,  0.1718,  0.1139,  0.0639, -0.0100,
          0.3484, -0.1868],
        [ 0.1051, -0.2273,  0.0248,  0.0659,  0.0625,  0.0928,  0.1973,  0.0027,
          0.1936, -0.0666]], grad_fn=<AddmmBackward0>)

In [5]:
#顺序块
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # module是Module子类的一个实例。把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

In [6]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[-0.2399,  0.0810,  0.2635,  0.0520, -0.1137, -0.1612,  0.1871, -0.0154,
          0.1883,  0.0881],
        [-0.1182,  0.0207,  0.2144, -0.0863, -0.0929, -0.1265,  0.1122,  0.0575,
          0.0603, -0.0422]], grad_fn=<AddmmBackward0>)

In [7]:
#实现一个FixedHiddenMLP类
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In [8]:
net = FixedHiddenMLP()
net(X)

tensor(0.1881, grad_fn=<SumBackward0>)

In [9]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(0.3581, grad_fn=<SumBackward0>)

一个块可以由许多层组成；一个块可以由许多块组成。

块可以包含代码。

块负责大量的内部处理，包括参数初始化和反向传播。

层和块的顺序连接由Sequential块处理。

## 参数管理


In [10]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))
net(X)

tensor([[-0.5814],
        [-0.5527]], grad_fn=<AddmmBackward0>)

In [11]:
#当通过Sequential类定义模型时， 可以通过索引来访问模型的任意层。
print(net[2].state_dict())

OrderedDict([('weight', tensor([[-0.1037, -0.3286,  0.1110, -0.0742, -0.1411,  0.1942, -0.2441, -0.2930]])), ('bias', tensor([-0.0830]))])


In [12]:
#目标参数
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([-0.0830], requires_grad=True)
tensor([-0.0830])


In [13]:
net[2].weight.grad == None

True

In [14]:
#一次性访问所有参数
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [15]:
#访问网络参数
net.state_dict()['2.bias'].data

tensor([-0.0830])

In [16]:
#从嵌套块收集参数
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 嵌套
        net.add_module(f'block {i}', block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[0.3026],
        [0.3026]], grad_fn=<AddmmBackward0>)

In [17]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [18]:
rgnet[0][1][0].bias.data

tensor([ 0.3337,  0.1678,  0.2467, -0.1499,  0.0800,  0.3890,  0.1632, -0.4856])

In [19]:
#内置初始化
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([-0.0085, -0.0048,  0.0010, -0.0032]), tensor(0.))

In [20]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [21]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([ 0.1512, -0.2085, -0.1977,  0.1359])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


In [22]:
#自定义初始化
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[ 7.4505, -0.0000, -9.5662,  0.0000],
        [-0.0000, -0.0000, -7.1765,  5.4385]], grad_fn=<SliceBackward0>)

In [23]:
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([42.0000,  1.0000, -8.5662,  1.0000])

In [24]:
#参数绑定
# 给共享层一个名称，以便可以引用它的参数
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
net(X)
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


## 延后初始化

延后初始化即直到数据第一次通过模型传递时，框架才会动态地推断出每个层的大小。  

延后初始化使框架能够自动推断参数形状，使修改模型架构变得容易，避免了一些常见的错误。

可以通过模型传递数据，使框架最终初始化参数。

## 自定义层

In [25]:
#不带参数的层
import torch
import torch.nn.functional as F
from torch import nn


class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

In [26]:
layer = CenteredLayer()
layer(torch.FloatTensor([1, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

In [27]:
net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())

In [28]:
Y = net(torch.rand(4, 8))
Y.mean()

tensor(2.9104e-09, grad_fn=<MeanBackward0>)

In [29]:
#带参数的层
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))
    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)

In [30]:
linear = MyLinear(5, 3)
linear.weight

Parameter containing:
tensor([[-1.5652,  1.5000,  0.4297],
        [ 2.1939, -0.5947, -1.3707],
        [ 0.5649,  0.4754,  0.7316],
        [ 1.7471, -0.5941,  0.0659],
        [ 0.6736, -2.1518,  0.7834]], requires_grad=True)

In [31]:
linear(torch.rand(2, 5))

tensor([[0.6263, 0.6752, 0.0000],
        [0.5216, 0.7002, 0.0000]])

In [32]:
net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))
net(torch.rand(2, 64))

tensor([[0.],
        [0.]])

在自定义层定义完成后，可以在任意环境和网络架构中调用该自定义层。

层可以有局部参数，这些参数可以通过内置函数创建。

## 读写文件


In [33]:
import torch
from torch import nn
from torch.nn import functional as F

x = torch.arange(4)
torch.save(x, 'x-file')

In [34]:
x2 = torch.load('x-file')
x2

tensor([0, 1, 2, 3])

In [35]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP()
X = torch.randn(size=(2, 20))
Y = net(X)

In [36]:
torch.save(net.state_dict(), 'mlp.params')

In [37]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
clone.eval()

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

In [38]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

save和load函数可用于张量对象的文件读写。  
可以通过参数字典保存和加载网络的全部参数。  
保存架构必须在代码中完成，而不是在参数中完成。

## GPU

使用nvidia-smi命令来查看显卡信息  
默认情况下，张量是在内存中创建的，然后使用CPU计算它。  
如果有多个GPU，使用torch.device(f'cuda:{i}') 来表示第i块GPU（i从0开始）。cuda:0和cuda是等价的。  


In [40]:
import torch
from torch import nn

torch.device('cpu'), torch.device('cuda'), torch.device('cuda:1')

(device(type='cpu'), device(type='cuda'), device(type='cuda', index=1))

使用torch.cuda.device_count()查询可用gpu的数量。

In [42]:
x = torch.tensor([1, 2, 3])
x.device

device(type='cpu')

要对多个项进行操作， 它们都必须在同一个设备上。   

可以指定用于存储和计算的设备，例如CPU或GPU。  

深度学习框架要求计算的所有输入数据都在同一设备上，无论是CPU还是GPU。